In [ ]:
!pip install -q transformers datasets accelerate sentencepiece indic-nlp-library openpyxl scikit-learn seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 10.4 MB/s eta 0:00:00


In [ ]:
import os
import re
import unicodedata
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
from indicnlp.tokenize import indic_tokenize

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

In [ ]:
print("PyTorch version:",torch.__version__)
print("CUDA available:",torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:",torch.cuda.get_device_name(0))
else:
    print("Training will use CPU.")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
train_df=pd.read_excel("../dataset/hate_bin_train.xlsx")
valid_df=pd.read_excel("../dataset/hate_bin_valid.xlsx")
test_df=pd.read_excel("../dataset/hate_bin_test.xlsx")

print("Train:",train_df.shape)
print("Validation:",valid_df.shape)
print("Test:",test_df.shape)

Train: (30000, 2)
Validation: (3750, 2)
Test: (3750, 2)


In [ ]:
print("Columns:",train_df.columns.tolist())
train_df.head()

Columns: ['text', 'label']


,text,label
0,@Rohitkh76543 @PawarSpeaks तू कोण लवड्या,HOF
1,@mohitbharatiya_ लवड्या अगोदरच रडायला लागलास तू 😂,HOF
2,"कुठं तरी आईघाल जा लवड्या, काही केला तरी तुझी क...",HOF
3,तेलतुंबड्या सारख्या च्या नाजायज पैदाइशीचा कुत्...,HOF
4,@TV9Marathi @BJP4Maharashtra नेता...???मादरचोद...,HOF


In [ ]:
print("TRAIN")
print(train_df["label"].value_counts())

print("\nVALIDATION")
print(valid_df["label"].value_counts())

print("\nTEST")
print(test_df["label"].value_counts())

TRAIN
label
HOF    15000
NOT    15000
Name: count, dtype: int64

VALIDATION
label
HOF    1875
NOT    1875
Name: count, dtype: int64

TEST
label
HOF    1875
NOT    1875
Name: count, dtype: int64


In [ ]:
normalizer_factory=IndicNormalizerFactory()
marathi_normalizer=normalizer_factory.get_normalizer("mr")

In [ ]:
def preprocess_marathi(text):
    if pd.isna(text):
        return ""

    text=str(text)

    # Unicode normalization
    text=unicodedata.normalize("NFC",text)

    # Normalize Marathi/Indic Unicode characters
    text=marathi_normalizer.normalize(text)

    # Remove zero-width characters
    text=re.sub(r"[\u200b\u200c\u200d\ufeff]","",text)

    # Convert English characters to lowercase
    text=text.lower()

    # Remove URLs because the project focuses on Marathi linguistic content
    text=re.sub(r"https?://\S+|www\.\S+"," ",text)

    # Remove social-media usernames
    text=re.sub(r"@\w+"," ",text)

    # Keep the word after # but remove only the hashtag symbol
    text=re.sub(r"#(\w+)",r"\1",text)

    # Reduce repeated punctuation
    text=re.sub(r"([!?।])\1+",r"\1",text)

    # Reduce excessive repetition of the same character
    text=re.sub(r"([^\W\d_])\1{3,}",r"\1\1",text)

    # Normalize multiple spaces
    text=re.sub(r"\s+"," ",text).strip()

    return text

In [ ]:
train_df["clean_text"]=train_df["text"].apply(preprocess_marathi)
valid_df["clean_text"]=valid_df["text"].apply(preprocess_marathi)
test_df["clean_text"]=test_df["text"].apply(preprocess_marathi)

In [ ]:
train_df["clean_text"]=train_df["text"].apply(preprocess_marathi)
valid_df["clean_text"]=valid_df["text"].apply(preprocess_marathi)
test_df["clean_text"]=test_df["text"].apply(preprocess_marathi)

In [ ]:
comparison=train_df[["text","clean_text","label"]].head()
display(comparison)

,text,clean_text,label
0,@Rohitkh76543 @PawarSpeaks तू कोण लवड्या,तू कोण लवड्या,HOF
1,@mohitbharatiya_ लवड्या अगोदरच रडायला लागलास तू 😂,लवड्या अगोदरच रडायला लागलास तू 😂,HOF
2,"कुठं तरी आईघाल जा लवड्या, काही केला तरी तुझी क...","कुठं तरी आईघाल जा लवड्या, काही केला तरी तुझी क...",HOF
3,तेलतुंबड्या सारख्या च्या नाजायज पैदाइशीचा कुत्...,तेलतुंबड्या सारख्या च्या नाजायज पैदाइशीचा कुत्...,HOF
4,@TV9Marathi @BJP4Maharashtra नेता...???मादरचोद...,नेता...?मादरचोद ची शहरप्रमुख व्हायची लायकी आहे...,HOF


In [ ]:
def marathi_word_tokenize(text):
    return indic_tokenize.trivial_tokenize(text,lang="mr")

In [ ]:
train_df["tokens"]=train_df["clean_text"].apply(marathi_word_tokenize)

In [ ]:
train_df[["text","clean_text","tokens","label"]].head()

,text,clean_text,tokens,label
0,@Rohitkh76543 @PawarSpeaks तू कोण लवड्या,तू कोण लवड्या,"[तू, कोण, लवड्या]",HOF
1,@mohitbharatiya_ लवड्या अगोदरच रडायला लागलास तू 😂,लवड्या अगोदरच रडायला लागलास तू 😂,"[लवड्या, अगोदरच, रडायला, लागलास, तू, 😂]",HOF
2,"कुठं तरी आईघाल जा लवड्या, काही केला तरी तुझी क...","कुठं तरी आईघाल जा लवड्या, काही केला तरी तुझी क...","[कुठं, तरी, आईघाल, जा, लवड्या, ,, काही, केला, ...",HOF
3,तेलतुंबड्या सारख्या च्या नाजायज पैदाइशीचा कुत्...,तेलतुंबड्या सारख्या च्या नाजायज पैदाइशीचा कुत्...,"[तेलतुंबड्या, सारख्या, च्या, नाजायज, पैदाइशीचा...",HOF
4,@TV9Marathi @BJP4Maharashtra नेता...???मादरचोद...,नेता...?मादरचोद ची शहरप्रमुख व्हायची लायकी आहे...,"[नेता, ., ., ., ?, मादरचोद, ची, शहरप्रमुख, व्ह...",HOF


In [ ]:
MODEL_NAME="ai4bharat/IndicBERTv2-MLM-only"

tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/639 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 7.75MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [ ]:
model=AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0:"NOT",1:"HOF"},
    label2id={"NOT":0,"HOF":1}
)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.12GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: ai4bharat/IndicBERTv2-MLM-only
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those p

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

In [ ]:
label_map={
    "NOT":0,
    "HOF":1
}

train_df["labels"]=train_df["label"].map(label_map)
valid_df["labels"]=valid_df["label"].map(label_map)
test_df["labels"]=test_df["label"].map(label_map)

In [ ]:
train_df[["label","labels"]].drop_duplicates()

,label,labels
0,HOF,1
15000,NOT,0


In [ ]:
train_dataset=Dataset.from_pandas(
    train_df[["clean_text","labels"]],
    preserve_index=False
)

valid_dataset=Dataset.from_pandas(
    valid_df[["clean_text","labels"]],
    preserve_index=False
)

test_dataset=Dataset.from_pandas(
    test_df[["clean_text","labels"]],
    preserve_index=False
)

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["clean_text"],
        truncation=True,
        max_length=128
    )

In [ ]:
train_dataset=train_dataset.map(
    tokenize_function,
    batched=True
)

valid_dataset=valid_dataset.map(
    tokenize_function,
    batched=True
)

test_dataset=test_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3750 [00:00<?, ? examples/s]

Map:   0%|          | 0/3750 [00:00<?, ? examples/s]

In [ ]:
sample=train_df.loc[0,"clean_text"]

print("Original sentence:")
print(train_df.loc[0,"text"])

print("\nPreprocessed sentence:")
print(sample)

print("\nMarathi word tokens:")
print(marathi_word_tokenize(sample))

print("\nIndicBERT subword tokens:")
print(tokenizer.tokenize(sample))

Original sentence:
@Rohitkh76543 @PawarSpeaks तू कोण लवड्या

Preprocessed sentence:
तू कोण लवड्या

Marathi word tokens:
['तू', 'कोण', 'लवड्या']

IndicBERT subword tokens:
['तू', 'कोण', 'लव', '##ड्या']


In [ ]:
data_collator=DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [ ]:
def compute_metrics(eval_pred):
    logits,labels=eval_pred

    predictions=np.argmax(logits,axis=1)

    accuracy=accuracy_score(
        labels,
        predictions
    )

    precision,recall,f1,_=precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy":accuracy,
        "precision":precision,
        "recall":recall,
        "f1":f1
    }

In [ ]:
training_args=TrainingArguments(
    output_dir="./marathi_hate_bert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    warmup_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available()
)

In [ ]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 3}.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.272498,0.231855,0.909067,0.910505,0.909067,0.908987
2,0.175304,0.233483,0.919200,0.919201,0.919200,0.919200


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3750, training_loss=0.24762488174438477, metrics={'train_runtime': 779.4175, 'train_samples_per_second': 76.981, 'train_steps_per_second': 4.811, 'total_flos': 2005004908488960.0, 'train_loss': 0.24762488174438477, 'epoch': 2.0})

In [ ]:
test_results=trainer.evaluate(test_dataset)

print("Test Accuracy:",round(test_results["eval_accuracy"],4))
print("Test Precision:",round(test_results["eval_precision"],4))
print("Test Recall:",round(test_results["eval_recall"],4))
print("Test F1:",round(test_results["eval_f1"],4))

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.175304,0.247094,2,0.914667,0.914735,0.914667,0.914663


Test Accuracy: 0.9147
Test Precision: 0.9147
Test Recall: 0.9147
Test F1: 0.9147


In [ ]:
test_output=trainer.predict(test_dataset)

test_predictions=np.argmax(
    test_output.predictions,
    axis=1
)

print(
    classification_report(
        test_df["labels"],
        test_predictions,
        target_names=["NOT","HOF"],
        digits=4
    )
)

              precision    recall  f1-score   support

         NOT     0.9200    0.9083    0.9141      1875
         HOF     0.9094    0.9211    0.9152      1875

    accuracy                         0.9147      3750
   macro avg     0.9147    0.9147    0.9147      3750
weighted avg     0.9147    0.9147    0.9147      3750



In [ ]:
cm=confusion_matrix(
    test_df["labels"],
    test_predictions
)

cm

array([[1703,  172],
       [ 148, 1727]])

In [ ]:
MODEL_PATH="./marathi_hate_bert_final"

trainer.save_model(MODEL_PATH)
tokenizer.save_pretrained(MODEL_PATH)

print("Model saved:",MODEL_PATH)
print(os.listdir(MODEL_PATH))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved: ./marathi_hate_bert_final
['tokenizer.json', 'model.safetensors', 'config.json', 'tokenizer_config.json', 'training_args.bin']


In [ ]:
def predict_text(text):
    cleaned_text=preprocess_marathi(text)

    inputs=tokenizer(
        cleaned_text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    device=next(model.parameters()).device

    inputs={
        key:value.to(device)
        for key,value in inputs.items()
    }

    model.eval()

    with torch.no_grad():
        outputs=model(**inputs)
        probabilities=torch.softmax(
            outputs.logits,
            dim=1
        )

    predicted_class=torch.argmax(
        probabilities,
        dim=1
    ).item()

    confidence=(
        probabilities[0][predicted_class].item()*100
    )

    label={
        0:"NOT",
        1:"HOF"
    }[predicted_class]

    if label=="HOF":
        if confidence>=80:
            risk="HIGH"
        elif confidence>=50:
            risk="MEDIUM"
        else:
            risk="LOW"
    else:
        risk="SAFE"

    return {
        "text":text,
        "prediction":label,
        "confidence":round(confidence,2),
        "risk":risk
    }

In [ ]:
print(predict_text("तू खूप वाईट माणूस आहेस"))

print(predict_text("आज हवामान खूप छान आहे"))

print(predict_text("तू नेहमी सगळ्यांना त्रास देतोस"))

print(predict_text("आज आपण मित्रांसोबत बाहेर जाणार आहोत"))

print(predict_text("तू काहीच कामाचा नाहीस, तुला काहीच कळत नाही."))


{'text': 'तू खूप वाईट माणूस आहेस', 'prediction': 'HOF', 'confidence': 97.61, 'risk': 'HIGH'}
{'text': 'आज हवामान खूप छान आहे', 'prediction': 'NOT', 'confidence': 99.85, 'risk': 'SAFE'}
{'text': 'तू नेहमी सगळ्यांना त्रास देतोस', 'prediction': 'NOT', 'confidence': 89.19, 'risk': 'SAFE'}
{'text': 'आज आपण मित्रांसोबत बाहेर जाणार आहोत', 'prediction': 'NOT', 'confidence': 99.82, 'risk': 'SAFE'}
{'text': 'तू काहीच कामाचा नाहीस, तुला काहीच कळत नाही.', 'prediction': 'HOF', 'confidence': 52.77, 'risk': 'MEDIUM'}


In [ ]:
import shutil

zip_path=shutil.make_archive(
    "/content/marathi_hate_bert_final",
    "zip",
    MODEL_PATH
)

print("ZIP created:")
print(zip_path)

ZIP created:
/content/marathi_hate_bert_final.zip


In [ ]:
from google.colab import files

files.download("/content/marathi_hate_bert_final.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>